# Transition Performance Index (TPI)

*Formerly the "decoupling index" - renamed because the score blends decoupling **rates** (how fast emissions fell) with current **states** (absolute footprint, grid cleanliness and prosperity). Decoupling is one input here, but not the whole index.*

**What this measures:** how well each European country combines an *honest*, consumption-based emissions reduction with a clean grid, a low absolute footprint and real prosperity. Countries was got scored from 0 to 100 against fixed real-world anchors, so adding or removing a country never reshuffles the others.

**What this does NOT claim:**
- A high rank doesn't mean "fast enough for Paris-compatible fair share" - it's relative. The **sufficiency overlay** below checks every country against the ~2 t/capita fair-share path.
- Scores inherit input uncertainty: consumption-based data carries roughly ±15–30% in the literature. So, I suggest to consider nearest countries as ties, and **Referee 6** will help with it.
- **Flags:** `!` = data-quality distortion (IRL, LUX, MLT - was excluded from the headline ranking). `*` = accounting-boundary caveat (NOR petrostate - kept in the ranking, because the data is accurate, but the accounting frame is kind to it).
- Scope is **Europe-only by design** - I know, it's relatively easy case (a deindustrialized and import-heavy continent), but I don't want to overkill myself.

Full methodology and the fairness-referee battery: see [`08_INDEX_METHODOLOGY.md`](08_INDEX_METHODOLOGY.md).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_END_YEAR

BASE_YEARS  = [1990, 2000]

# Normative weights (must sum to 1.0). 
# Honesty got 0.40 - the thesis carries the most weight; the rest share equally.
WEIGHTS = {
    'honesty':         0.40,   # consumption CO2 reduction minus fake-decoupling penalty (can be negative)
    'energy_clean':    0.15,   # absolute current grid cleanliness (rewards already-clean countries)
    'abs_consumption': 0.15,   # absolute consumption CO2/capita vs 2t target
    'prosperity':      0.15,   # absolute GDP/capita + positive-growth guard
    'co2_reduction':   0.15,   # territorial CO2/capita reduction
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

# Sub-weights inside the prosperity component
PROSPERITY_SUB = {'abs_gdp': 0.6, 'growth_guard': 0.4}

# Honesty = blend of long-run and recent consumption reduction, minus fake penalty
HONESTY_BLEND = {'long': 0.6, 'recent': 0.4}

MOMENTUM = {'window_start': 2010, 'full_swing_slope': 0.30, 'max_bonus': 3.0}

# 2t = Paris-compatible fair share
PARIS = {'target_t': 2.0, 'target_year': 2050, 'rate_from': 2010}

# Criterion-referenced scoring thresholds (fixed, cohort-independent)
# 45% consumption cut treated as comparably demanding (consumption includes imports)
# 0.03 kgCO2/kWh = near-zero-carbon grid
# 55 % of co2 reduction - EU Fit-for-55 ambition
SCORING = {
    'co2_reduction':         {'zero': 0.0,    'full': 55.0},
    'cons_reduction':        {'zero': 0.0,    'full': 45.0},
    'cons_reduction_recent': {'zero': 0.0,    'full': 30.0},
    'fake_penalty':          {'zero': 0.0,    'full': 60.0},   # if terr fell more than cons -> penalty
    'abs_consumption':       {'zero': 16.0,   'full': PARIS['target_t']},
    'energy_clean':          {'zero': 0.30,   'full': 0.03},   
    'abs_gdp':               {'zero': 15000.0,'full': 60000.0},
    'growth_guard':          {'zero': -30.0,  'full': 0.0},
}



# Data-quality distortions: flagged '!' and EXCLUDED from the headline ('clean') ranking
DATA_CAVEAT = {
    'IRL': 'GDP inflated by multinational profit-shifting (real income ~ GNI*)',
    'LUX': 'cross-border workers inflate per-capita GDP & emissions; fuel-tourism legacy',
    'MLT': 'tiny island economy, sparse/volatile data',
}

# Accounting-boundary caveats: flagged '*' but KEPT in the ranking - the data is accurate,
# the accounting frame is kind to them. Flag, don't punish, until it can be quantified fairly.
# To quantify (future work, would need ALL of Europe for fairness): extraction-based accounting
# from the OWID energy dataset (owid-energy-data.csv: oil_production / gas_production /
# coal_production, TWh) x IPCC combustion emission factors. Underlying source: Energy Institute
# Statistical Review; policy framing: UNEP Production Gap Report.
BOUNDARY_CAVEAT = {
    'NOR': 'petrostate: emissions of exported oil & gas sit outside BOTH territorial and '
           'consumption accounting - the clean domestic image is funded by combustion abroad',
}

# Decoupling verdicts from NB04 (coloring / face-validity)
GENUINE = ['DEU', 'SWE', 'ROU', 'EST', 'LTU', 'SVK', 'BGR', 'FIN', 'NOR']
FAKE    = ['CHE', 'BEL', 'DNK', 'FRA', 'GBR', 'BLR']
SPECIAL = ['POL', 'IRL', 'CYP', 'GEO']

def verdict_of(iso):
    if iso in GENUINE: return 'Genuine'
    if iso in FAKE:    return 'Fake'
    if iso in SPECIAL: return 'Special'
    return 'Other'

VERDICT_COLORS = {'Genuine': '#2ecc71', 'Fake': '#e74c3c',
                  'Special': '#f39c12', 'Other': '#95a5a6'}

print('Config loaded. Weights sum =', round(sum(WEIGHTS.values()), 4))

Config loaded. Weights sum = 1.0


In [2]:
q = """
    SELECT
        c.name AS country, 
        c.iso_code, 
        e.year,
        e.co2_per_capita,
        e.consumption_co2_per_capita,
        e.co2_per_unit_energy,
        e.gdp, e.population
    FROM 
        emissions e
    JOIN 
        countries c ON c.id = e.country_id
    WHERE 
        c.iso_code = ANY(:countries)
    ORDER BY 
        c.iso_code, 
        e.year
"""
with engine.connect() as conn:
    df = pd.read_sql(
        text(q), 
        conn,
        params={'countries': EUROPEAN_COUNTRIES}
    )

df['gdp_per_capita'] = df['gdp'] / df['population']
print(f"Loaded {len(df)} rows, {df['iso_code'].nunique()} countries")

latest = df.sort_values('year').groupby('iso_code')
print("\nLatest co2_per_unit_energy (kgCO2/kWh) - calibrate 'energy_clean':")
print(latest['co2_per_unit_energy'].last().describe().round(3).to_string())
print("\nLatest gdp_per_capita - calibrate 'abs_gdp':")
print(latest['gdp_per_capita'].last().describe().round(0).to_string())
print("\nLatest consumption_co2_per_capita - calibrate 'abs_consumption':")
print(latest['consumption_co2_per_capita'].last().describe().round(2).to_string())

2026-06-17 00:08:57,551 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-06-17 00:08:57,552 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-17 00:08:57,556 INFO sqlalchemy.engine.Engine select current_schema()
2026-06-17 00:08:57,557 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-17 00:08:57,559 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-06-17 00:08:57,561 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-17 00:08:57,569 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 00:08:57,570 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname

In [3]:
def value_at_year(country_ts, year, col):
    exact_year = country_ts[country_ts['year'] == year]
    return np.nan if exact_year.empty else exact_year[col].values[0]

def latest_value(country_ts, col, not_after=DECOUPLING_END_YEAR):
    observed = country_ts[(country_ts['year'] <= not_after) & country_ts[col].notna()]
    return np.nan if observed.empty else observed.sort_values('year')[col].values[-1]

def pct_reduction(start_value, end_value):
    # positive = the value FELL
    if pd.isna(start_value) or pd.isna(end_value) or start_value == 0: 
        return np.nan
    return (start_value - end_value) / start_value * 100

def pct_growth(start_value, end_value):
    if pd.isna(start_value) or pd.isna(end_value) or start_value == 0: 
        return np.nan
    return (end_value - start_value) / start_value * 100

def slope_per_year_since(country_ts, col, start_year):
    min_observations = 4
    window = country_ts[(country_ts['year'] >= start_year) & country_ts[col].notna()]
    if len(window) < min_observations: 
        return np.nan
    return np.polyfit(window['year'].astype(float), window[col].astype(float), 1)[0]

In [4]:
def score_linear(value, zero, full):
    if pd.isna(value): 
        return np.nan
    return float(np.clip((value - zero) / (full - zero) * 100, 0, 100))

def momentum_bonus(slope):
    # falling consumption (negative slope) -> positive bonus, capped at +-max_bonus
    if pd.isna(slope): 
        return 0.0
    raw_bonus = -slope / MOMENTUM['full_swing_slope'] * MOMENTUM['max_bonus']
    return float(np.clip(raw_bonus, -MOMENTUM['max_bonus'], MOMENTUM['max_bonus']))

def weighted_avg_available(component_scores, weights):
    # weights renormalized over what's present
    available = {name: score for name, score in component_scores.items() if pd.notna(score)}
    if not available: 
        return np.nan
    total_weight = sum(weights[name] for name in available)
    return sum(score * weights[name] for name, score in available.items()) / total_weight

In [5]:
# ACTIVE_SCORING is the anchor set build_scores reads. Referee 4 temporarily swaps it
# for a jittered copy, so build_scores must re-read it on every call.
ACTIVE_SCORING = SCORING

def honesty_base_year(country_ts, base_year):
    both_present = country_ts[(country_ts['year'] >= base_year) &
                              country_ts['consumption_co2_per_capita'].notna() &
                              country_ts['co2_per_capita'].notna()]
    return int(both_present['year'].min()) if not both_present.empty else None

def first_value_since(country_ts, col, start_year):
    observed = country_ts[(country_ts['year'] >= start_year) & country_ts[col].notna()].sort_values('year')
    return observed[col].values[0] if not observed.empty else np.nan

def build_scores(df, base_year):
    anchors = ACTIVE_SCORING
    score_rows = []
    for iso, country_ts in df.groupby('iso_code'):
        country_ts = country_ts.sort_values('year')
        country = country_ts['country'].iloc[0]

        territorial_at_base = value_at_year(country_ts, base_year, 'co2_per_capita')
        territorial_now = latest_value(country_ts, 'co2_per_capita')
        consumption_now = latest_value(country_ts, 'consumption_co2_per_capita')
        gdp_pc_at_base = value_at_year(country_ts, base_year, 'gdp_per_capita')
        gdp_pc_now = latest_value(country_ts, 'gdp_per_capita')
        grid_intensity_now = latest_value(country_ts, 'co2_per_unit_energy')

        territorial_cut_pct = pct_reduction(territorial_at_base, territorial_now)  # from GLOBAL base year
        gdp_growth_pct = pct_growth(gdp_pc_at_base, gdp_pc_now)
        if pd.notna(consumption_now) and pd.notna(territorial_now) and territorial_now != 0:
            gap_pct = (consumption_now - territorial_now) / territorial_now * 100
        else:
            gap_pct = np.nan

        honesty_start_year = honesty_base_year(country_ts, base_year)
        if honesty_start_year is not None:
            consumption_at_honesty_start = value_at_year(country_ts, honesty_start_year, 'consumption_co2_per_capita')
            territorial_at_honesty_start = value_at_year(country_ts, honesty_start_year, 'co2_per_capita')
            consumption_cut_pct = pct_reduction(consumption_at_honesty_start, consumption_now)
            territorial_cut_from_honesty_start = pct_reduction(territorial_at_honesty_start, territorial_now)
            divergence = (territorial_cut_from_honesty_start - consumption_cut_pct) \
                if (pd.notna(territorial_cut_from_honesty_start) and pd.notna(consumption_cut_pct)) else np.nan
        else:
            consumption_cut_pct = divergence = np.nan

        consumption_at_recency_start = first_value_since(country_ts, 'consumption_co2_per_capita', 2010)
        consumption_cut_recent_pct = pct_reduction(consumption_at_recency_start, consumption_now)

        # honesty: blend(long, recent) consumption reduction minus fake penalty (can go negative)
        long_run_score = score_linear(consumption_cut_pct,
                                      anchors['cons_reduction']['zero'], anchors['cons_reduction']['full'])
        recent_score = score_linear(consumption_cut_recent_pct,
                                      anchors['cons_reduction_recent']['zero'], anchors['cons_reduction_recent']['full'])
        honesty_before_penalty = weighted_avg_available(
            {'long': long_run_score, 'recent': recent_score}, HONESTY_BLEND)
        fake_decoupling_penalty = score_linear(divergence,
                                               anchors['fake_penalty']['zero'], anchors['fake_penalty']['full'])
        fake_decoupling_penalty = 0.0 if pd.isna(fake_decoupling_penalty) else fake_decoupling_penalty
        s_honesty = (honesty_before_penalty - fake_decoupling_penalty) \
            if pd.notna(honesty_before_penalty) else np.nan

        s_energy = score_linear(grid_intensity_now,
                                anchors['energy_clean']['zero'], anchors['energy_clean']['full'])

        s_abs = score_linear(
            consumption_now,
            anchors['abs_consumption']['zero'], 
            anchors['abs_consumption']['full']
        )

        abs_gdp_score = score_linear(gdp_pc_now, anchors['abs_gdp']['zero'], anchors['abs_gdp']['full'])
        growth_guard_score = score_linear(gdp_growth_pct, anchors['growth_guard']['zero'], anchors['growth_guard']['full'])
        s_prosperity = weighted_avg_available(
            {'abs_gdp': abs_gdp_score, 'growth_guard': growth_guard_score}, PROSPERITY_SUB)

        s_co2 = score_linear(
            territorial_cut_pct,
            anchors['co2_reduction']['zero'], 
            anchors['co2_reduction']['full']
        )

        component_scores = {
            'honesty': s_honesty, \
            'energy_clean': s_energy, 
            'abs_consumption': s_abs,
            'prosperity': s_prosperity, 
            'co2_reduction': s_co2
        }
        composite = weighted_avg_available(component_scores, WEIGHTS)

        momentum_pts = momentum_bonus(
            slope_per_year_since(country_ts, 'consumption_co2_per_capita', MOMENTUM['window_start']))
        final = composite + momentum_pts if pd.notna(composite) else np.nan

        score_rows.append({
            'iso_code': iso, 'country': country, 'verdict': verdict_of(iso),
            # '!' = data-quality (excluded from clean ranking); '*' = boundary caveat (kept, read with care)
            'flag': ('!' if iso in DATA_CAVEAT else '') + ('*' if iso in BOUNDARY_CAVEAT else ''),
            'honesty_base_yr': honesty_start_year,
            'co2_red_pct': territorial_cut_pct, 'cons_red_pct': consumption_cut_pct,
            'cons_red_recent_pct': consumption_cut_recent_pct,
            'gdp_grw_pct': gdp_growth_pct, 'gap_pct': gap_pct, 'divergence': divergence,
            'cons_latest': consumption_now, 'gdp_latest': gdp_pc_now, 'en_latest': grid_intensity_now,
            's_honesty': s_honesty, 's_energy': s_energy, 's_abs': s_abs,
            's_prosperity': s_prosperity, 's_co2': s_co2,
            'composite': composite, 'momentum': momentum_pts, 'final': final,
        })

    scores_table = pd.DataFrame(score_rows)
    # gate: must have territorial reduction + absolute consumption
    scores_table = scores_table[scores_table['final'].notna() &
                                scores_table['s_co2'].notna() &
                                scores_table['s_abs'].notna()].copy()
    return scores_table.sort_values('final', ascending=False).reset_index(drop=True)

scores_2000 = build_scores(df, 2000)
scores_1990 = build_scores(df, 1990)

print(f"2000 base: {len(scores_2000)} countries scored")
print(f"1990 base: {len(scores_1990)} countries scored")

for base_label, scores_table in [(2000, scores_2000), (1990, scores_1990)]:
    late_starters = scores_table[scores_table['honesty_base_yr'].notna() &
                                 (scores_table['honesty_base_yr'] != base_label)]
    if len(late_starters):
        print(f"\n[{base_label} base] honesty measured from a later year (consumption data starts late):")
        for _, row in late_starters.sort_values('honesty_base_yr').iterrows():
            print(f"   {row['iso_code']}: honesty base {int(row['honesty_base_yr'])}")

2000 base: 34 countries scored
1990 base: 34 countries scored

[2000 base] honesty measured from a later year (consumption data starts late):
   NOR: honesty base 2003

[1990 base] honesty measured from a later year (consumption data starts late):
   NOR: honesty base 2003


In [6]:
def clean_subset(scores):
    # headline ranking: drop data-quality distortions ('!'), KEEP boundary-flagged ('*')
    return scores[~scores['flag'].str.contains('!')]

def show_ranking(scores, base_year, n=15, exclude_flagged=False):
    ranking = clean_subset(scores) if exclude_flagged else scores
    display_columns = ['flag', 'iso_code', 'country', 'final', 'composite', 'momentum',
                       's_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
    display_table = ranking.head(n)[display_columns].copy()
    display_table.insert(0, 'rank', range(1, len(display_table) + 1))
    numeric_columns = ['final', 'composite', 'momentum',
                       's_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
    for col in numeric_columns:
        display_table[col] = display_table[col].round(1)
    subtitle = '  (excl. data-quality flags)' if exclude_flagged else ''
    print(f"TRANSITION PERFORMANCE INDEX (base {base_year}){subtitle} - Top {n}")
    print(display_table.to_string(index=False))

show_ranking(scores_2000, 2000, exclude_flagged=True)
print()
show_ranking(scores_2000, 2000)
print()
show_ranking(scores_1990, 1990, exclude_flagged=True)

print("\n'!' data-quality caveats (excluded from the headline ranking):")
for iso, why in DATA_CAVEAT.items():
    print(f"   {iso}: {why}")
print("\n'*' accounting-boundary caveats (kept in the ranking - read with care):")
for iso, why in BOUNDARY_CAVEAT.items():
    print(f"   {iso}: {why}")

TRANSITION PERFORMANCE INDEX (base 2000)  (excl. data-quality flags) - Top 15
 rank flag iso_code        country  final  composite  momentum  s_honesty  s_energy  s_abs  s_prosperity  s_co2
    1           SWE         Sweden   83.6       81.5       2.1       82.4      88.5   72.5          82.8   79.6
    2           PRT       Portugal   83.2       81.6       1.6      100.0      58.9   81.8          57.8   78.9
    3           FIN        Finland   83.0       80.0       3.0       90.5      75.9   53.8          74.1   88.1
    4    *      NOR         Norway   81.0       78.0       3.0       84.7      85.2   63.8         100.0   45.3
    5           GBR United Kingdom   77.7       74.9       2.8       80.9      51.9   63.6          71.1   97.2
    6           EST        Estonia   73.6       70.6       3.0       87.0      53.7   48.5          59.5   76.8
    7           DNK        Denmark   69.3       67.0       2.3       57.3      56.3   54.9          87.6   95.2
    8           DEU       

In [7]:
# Horizontal bar chart of the headline (clean) ranking - top 15
def tpi_barchart(scores, base_year):
    top15 = clean_subset(scores).head(15).copy()
    top15['label'] = top15['country'] + np.where(top15['flag'].str.contains(r'\*'), '*', '')
    top15 = top15.sort_values('final')

    fig = px.bar(
        top15,
        x='final', y='label',
        orientation='h',
        color='final',
        color_continuous_scale='Tealgrn',
        text='final',
        title=f'Transition Performance Index - Top 15 ({base_year} base, data-quality flags excluded)<br>'
              '<sup>* = accounting-boundary caveat (e.g. Norway: exported oil & gas outside the frame)</sup>',
        labels={'final': 'TPI score', 'label': ''},
        height=600,
    )
    fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
    fig.update_layout(coloraxis_showscale=False, xaxis_range=[0, 100])
    fig.show()

tpi_barchart(scores_2000, 2000)
tpi_barchart(scores_1990, 1990)

In [ ]:
ranked_2000 = scores_2000.reset_index(drop=True).copy()
ranked_1990 = scores_1990.reset_index(drop=True).copy()
ranked_2000['rank_2000'] = range(1, len(ranked_2000) + 1)
ranked_1990['rank_1990'] = range(1, len(ranked_1990) + 1)

rank_shift = ranked_2000[['iso_code', 'country', 'verdict', 'rank_2000']].merge(
    ranked_1990[['iso_code', 'rank_1990']], on='iso_code', how='inner'
)
rank_shift['shift'] = rank_shift['rank_2000'] - rank_shift['rank_1990']  # +ve = worse under 2000

fig = go.Figure()
for _, country_row in rank_shift.iterrows():
    fig.add_trace(go.Scatter(
        x=['1990 base', '2000 base'],
        y=[country_row['rank_1990'], country_row['rank_2000']],
        mode='lines+markers+text',
        text=[country_row['country'], country_row['country']],
        textposition='middle right',
        line=dict(color=VERDICT_COLORS.get(country_row['verdict'], '#95a5a6')),
        showlegend=False
    ))
fig.update_layout(
    title='Ranking shift: 1990 vs 2000 baseline<br>'
          '<sup>Lines dropping left to right rode the Soviet-collapse windfall</sup>',
    yaxis=dict(title='Rank', autorange='reversed'),
    height=750
)
fig.show()

print("Most baseline-sensitive (rank worsens most under 2000 = inflated by 1990 collapse):")
print(rank_shift.sort_values('shift', ascending=False)
      [['country', 'verdict', 'rank_1990', 'rank_2000', 'shift']].head(8).to_string(index=False))

Most baseline-sensitive (rank worsens most under 2000 = inflated by 1990 collapse):
    country verdict  rank_1990  rank_2000  shift
Netherlands   Other          9         22     13
   Slovakia Genuine          4         17     13
    Romania Genuine         16         28     12
    Ukraine   Other         21         31     10
    Belarus    Fake         30         34      4
    Germany Genuine          6         10      4
  Lithuania Genuine         23         27      4
 Luxembourg   Other          3          6      3


In [9]:
top15 = scores_2000.head(15).copy()
contributions = pd.DataFrame({'iso_code': top15['iso_code']})
component_pairs = [('honesty', 's_honesty'), ('co2_reduction', 's_co2'),
                   ('abs_consumption', 's_abs'), ('energy_clean', 's_energy'),
                   ('prosperity', 's_prosperity')]
for component_name, score_col in component_pairs:
    contributions[component_name] = (top15[score_col].fillna(0) * WEIGHTS[component_name]).values

legend_labels = {'honesty': 'Honesty (cons. reduction - fake penalty)',
                 'co2_reduction': 'Territorial CO2 reduction',
                 'abs_consumption': 'Absolute consumption',
                 'energy_clean': 'Energy cleanliness (absolute)',
                 'prosperity': 'Prosperity (GDP)'}
component_colors = {'honesty': '#1abc9c', 'co2_reduction': '#3498db', 'abs_consumption': '#9b59b6',
                    'energy_clean': '#e67e22', 'prosperity': '#2c3e50'}

fig = go.Figure()
for component_name, _ in component_pairs:
    fig.add_trace(go.Bar(name=legend_labels[component_name],
                         x=contributions['iso_code'], y=contributions[component_name],
                         marker_color=component_colors[component_name]))
fig.update_layout(
    barmode='relative', height=560,
    title='What drives each score? Weighted contributions (2000 base, Top 15)<br>'
          '<sup>Honesty can dip below zero for fake decouplers</sup>',
    yaxis_title='Points (weighted)', xaxis_title='Country'
)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.show()

In [10]:
journey_vs_destination = scores_2000.copy()
fig = px.scatter(
    journey_vs_destination,
    x='composite',
    y='cons_latest',
    color='verdict',
    text='iso_code',
    color_discrete_map=VERDICT_COLORS,
    title='Journey vs Destination (2000 base)<br>'
          '<sup>X = TPI score | Y = consumption CO2/capita now. Top-right = clean AND high-scoring.</sup>',
    labels={'composite': 'TPI score', 'cons_latest': 'Consumption CO2 per capita (t, latest)'},
    height=680
)
fig.add_hline(y=2, line_dash='dash', line_color='green', annotation_text='2t target')
fig.update_traces(textposition='top center', marker=dict(size=11))
fig.update_yaxes(autorange='reversed')
fig.show()

In [11]:
# ---------- SUFFICIENCY OVERLAY: is anyone actually fast enough? ----------
# The index ranks countries against EACH OTHER (relative virtue). This overlay checks them
# against PHYSICS: at each country's real consumption-CO2 pace, when does it reach the
# ~2 t/cap Paris-compatible fair share? NOT part of the score - it reframes the whole ranking.

FAIR_SHARE_T      = PARIS['target_t']      # ~2 t consumption CO2/capita
DEADLINE_YEAR     = PARIS['target_year']   # 2050
PACE_WINDOW_START = PARIS['rate_from']     # measure actual pace from 2010

sufficiency_rows = []
for iso, country_ts in df.groupby('iso_code'):
    country_ts = country_ts.sort_values('year')
    consumption_ts = country_ts[country_ts['consumption_co2_per_capita'].notna()]
    if consumption_ts.empty:
        continue
    latest_year = int(consumption_ts['year'].max())
    consumption_now = float(consumption_ts.loc[consumption_ts['year'] == latest_year,
                                               'consumption_co2_per_capita'].iloc[0])
    pace_window = consumption_ts[consumption_ts['year'] >= PACE_WINDOW_START]
    if len(pace_window) < 4 or consumption_now <= 0:
        continue
    pace_start_year = int(pace_window['year'].iloc[0])
    consumption_at_pace_start = float(pace_window['consumption_co2_per_capita'].iloc[0])
    years_observed = latest_year - pace_start_year
    if years_observed < 4 or consumption_at_pace_start <= 0:
        continue

    # compound annual rates; negative = falling
    actual_rate = (consumption_now / consumption_at_pace_start) ** (1 / years_observed) - 1
    years_to_deadline = DEADLINE_YEAR - latest_year
    required_rate = ((FAIR_SHARE_T / consumption_now) ** (1 / years_to_deadline) - 1
                     if consumption_now > FAIR_SHARE_T else 0.0)

    if consumption_now <= FAIR_SHARE_T:
        arrival_estimate, verdict = latest_year, 'already at fair share'
    elif actual_rate < 0:
        arrival_estimate = latest_year + np.log(FAIR_SHARE_T / consumption_now) / np.log(1 + actual_rate)
        verdict = 'on track' if arrival_estimate <= DEADLINE_YEAR else 'too slow'
    else:
        arrival_estimate, verdict = np.inf, 'moving away'

    sufficiency_rows.append({
        'iso_code': iso, 'country': country_ts['country'].iloc[0],
        'cons_now_t': round(consumption_now, 1),
        'actual_pct_yr': round(actual_rate * 100, 2),      # negative = cutting
        'required_pct_yr': round(required_rate * 100, 2),  # pace needed for 2t by 2050
        'arrival_year': int(round(arrival_estimate)) if np.isfinite(arrival_estimate) else None,
        'verdict': verdict,
    })

sufficiency = pd.DataFrame(sufficiency_rows)
verdict_order = {'already at fair share': 0, 'on track': 1, 'too slow': 2, 'moving away': 3}
sufficiency['_verdict_rank'] = sufficiency['verdict'].map(verdict_order)
sufficiency['_arrival_sort'] = sufficiency['arrival_year'].fillna(9999)
sufficiency = (sufficiency.sort_values(['_verdict_rank', '_arrival_sort'])
               .drop(columns=['_verdict_rank', '_arrival_sort'])
               .reset_index(drop=True))

print(f"SUFFICIENCY CHECK - reach {FAIR_SHARE_T}t consumption CO2/capita by {DEADLINE_YEAR}")
print(f"(actual pace measured {PACE_WINDOW_START} -> latest, compound annual rate)\n")
print(sufficiency.to_string(index=False))

on_track_count = sufficiency['verdict'].isin(['already at fair share', 'on track']).sum()
print(f"\n>>> {on_track_count} of {len(sufficiency)} European countries are on a Paris-compatible path. <<<")

# the kicker: even the index leaders
for iso in clean_subset(scores_2000).head(3)['iso_code']:
    leader_match = sufficiency[sufficiency['iso_code'] == iso]
    if not leader_match.empty:
        leader_pace = leader_match.iloc[0]
        arrival_text = leader_pace['arrival_year'] if leader_pace['arrival_year'] else 'never at current pace'
        print(f"    index leader {iso}: doing {leader_pace['actual_pct_yr']:+.1f}%/yr, "
              f"needs {leader_pace['required_pct_yr']:+.1f}%/yr -> 2t reached ~{arrival_text}")

# Chart: actual vs required pace for the index top-12 (clean)
top12_codes = clean_subset(scores_2000).head(12)['iso_code'].tolist()
pace_df = sufficiency[sufficiency['iso_code'].isin(top12_codes)].copy()
pace_df['Actual cut %/yr']   = -pace_df['actual_pct_yr']    # positive = cutting
pace_df['Required cut %/yr'] = -pace_df['required_pct_yr']
pace_df['tpi_order'] = pace_df['iso_code'].map({iso: i for i, iso in enumerate(top12_codes)})
pace_df = pace_df.sort_values('tpi_order')

fig = go.Figure()
fig.add_trace(go.Bar(name=f'Actual pace ({PACE_WINDOW_START}->latest)', x=pace_df['iso_code'],
                     y=pace_df['Actual cut %/yr'], marker_color='#3498db'))
fig.add_trace(go.Bar(name=f'Required for {FAIR_SHARE_T}t by {DEADLINE_YEAR}', x=pace_df['iso_code'],
                     y=pace_df['Required cut %/yr'], marker_color='#e74c3c'))
fig.update_layout(
    barmode='group', height=520,
    title='Sufficiency: actual vs required pace of consumption-CO2 cuts - index Top 12<br>'
          '<sup>Blue below red = high TPI rank, still not Paris-compatible. Relative virtue is not sufficiency.</sup>',
    yaxis_title='Annual reduction rate (%/yr, positive = cutting)'
)
fig.show()

SUFFICIENCY CHECK - reach 2.0t consumption CO2/capita by 2050
(actual pace measured 2010 -> latest, compound annual rate)

iso_code        country  cons_now_t  actual_pct_yr  required_pct_yr  arrival_year               verdict
     ALB        Albania         1.9          -1.23             0.00        2023.0 already at fair share
     PRT       Portugal         4.5          -5.39            -3.00        2038.0              on track
     FIN        Finland         8.5          -4.89            -5.21        2052.0              too slow
     NOR         Norway         7.1          -4.26            -4.56        2052.0              too slow
     SWE         Sweden         5.8          -3.16            -3.90        2056.0              too slow
     GBR United Kingdom         7.1          -3.25            -4.58        2061.0              too slow
     GRC         Greece         6.6          -3.00            -4.30        2062.0              too slow
     LUX     Luxembourg        11.0          

In [12]:
# ---------- CONTEXT: historical responsibility (the stock, not the flow) ----------
# The atmosphere integrates over ~150 years; the index only sees recent flows.
# Cumulative emissions are shown as CONTEXT - never scored. The stock doesn't forget.
# Source: OWID cumulative_co2 (Mt since 1750) - read from the cached raw CSV (not in our DB schema).

csv_candidates = [os.path.join('..', 'data', 'owid-co2-data.csv'),
                  os.path.join('data', 'owid-co2-data.csv')]
owid_path = next((path for path in csv_candidates if os.path.exists(path)),
                 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv')
print(f"Reading cumulative columns from: {owid_path}")

owid_raw = pd.read_csv(owid_path,
                       usecols=['iso_code', 'year', 'cumulative_co2',
                                'share_global_cumulative_co2', 'population'])
europe_cumulative = owid_raw[owid_raw['iso_code'].isin(EUROPEAN_COUNTRIES) &
                             owid_raw['cumulative_co2'].notna()]
latest_cumulative = europe_cumulative.sort_values('year').groupby('iso_code').tail(1).copy()
latest_cumulative['cum_per_capita_t'] = (latest_cumulative['cumulative_co2'] * 1e6
                                         / latest_cumulative['population'])

stock_vs_flow = clean_subset(scores_2000)[['iso_code', 'country', 'final', 'flag']].merge(
    latest_cumulative[['iso_code', 'cumulative_co2', 'share_global_cumulative_co2', 'cum_per_capita_t']],
    on='iso_code', how='left')
stock_vs_flow = stock_vs_flow.rename(columns={'cumulative_co2': 'cum_total_Mt',
                                              'share_global_cumulative_co2': 'global_share_pct'})
stock_vs_flow.insert(0, 'tpi_rank', range(1, len(stock_vs_flow) + 1))

print("\nTPI rank vs historical responsibility (context, NOT scored):")
display_table = stock_vs_flow.head(15).copy()
display_table['cum_total_Mt']     = display_table['cum_total_Mt'].round(0)
display_table['global_share_pct'] = display_table['global_share_pct'].round(2)
display_table['cum_per_capita_t'] = display_table['cum_per_capita_t'].round(0)
print(display_table[['tpi_rank', 'flag', 'country', 'final', 'cum_total_Mt',
                     'global_share_pct', 'cum_per_capita_t']].round(1).to_string(index=False))

print("\nLargest historical debts per person (cumulative tonnes/capita):")
print(stock_vs_flow.sort_values('cum_per_capita_t', ascending=False)
      [['country', 'tpi_rank', 'cum_per_capita_t']].head(8).round(0).to_string(index=False))

fig = px.scatter(
    stock_vs_flow, x='final', y='cum_per_capita_t', text='iso_code',
    title='Today\'s performance vs yesterday\'s debt<br>'
          '<sup>X = TPI score (flow) | Y = cumulative CO2 per capita since 1750 (stock). '
          'Top-right = leads today AND owes the most history.</sup>',
    labels={'final': 'TPI score (2000 base)', 'cum_per_capita_t': 'Cumulative CO2 (t/capita, since 1750)'},
    height=600,
)
fig.update_traces(textposition='top center', marker=dict(size=11, color='#16a085'))
fig.show()

Reading cumulative columns from: ..\data\owid-co2-data.csv

TPI rank vs historical responsibility (context, NOT scored):
 tpi_rank flag        country  final  cum_total_Mt  global_share_pct  cum_per_capita_t
        1              Sweden   83.6        5135.0               0.3             484.0
        2            Portugal   83.2        2768.0               0.2             265.0
        3             Finland   83.0        3325.0               0.2             592.0
        4    *         Norway   81.0        2787.0               0.2             500.0
        5      United Kingdom   77.7       80079.0               4.3            1158.0
        6             Estonia   73.6        1668.0               0.1            1226.0
        7             Denmark   69.3        4200.0               0.2             703.0
        8             Germany   68.2       95136.0               5.1            1125.0
        9               Spain   67.0       15700.0               0.8             328.0
       10

In [13]:
# SENSITIVITY 1: Named weighting scenarios
component_columns = {'honesty': 's_honesty', 'energy_clean': 's_energy', 'abs_consumption': 's_abs',
                     'prosperity': 's_prosperity', 'co2_reduction': 's_co2'}

def rank_with_weights(scores_df, weights):
    # build the score dict ONLY over the components present in `weights` -
    # leave-one-out drops a key, and weighted_avg_available must never see a
    # component whose weight is missing (this was the KeyError)
    def composite_under_weights(row):
        return weighted_avg_available(
            {name: row[component_columns[name]] for name in weights}, weights)
    reranked = scores_df.copy()
    reranked['test_score'] = reranked.apply(composite_under_weights, axis=1)
    reranked = reranked.sort_values('test_score', ascending=False).reset_index(drop=True)
    return {row['iso_code']: position + 1 for position, row in reranked.iterrows()}

# basis = CLEAN ranking (data-quality flags excluded) - the headline leaderboard
clean_scores_2000 = clean_subset(scores_2000).reset_index(drop=True)

scenarios = {
    'Normative':         WEIGHTS,
    'Equal':             {name: 0.20 for name in WEIGHTS},
    'Honesty-heavy':     {'honesty': 0.60, 'energy_clean': 0.10, 'abs_consumption': 0.10, 'prosperity': 0.10, 'co2_reduction': 0.10},
    'Destination-heavy': {'honesty': 0.20, 'energy_clean': 0.25, 'abs_consumption': 0.30, 'prosperity': 0.15, 'co2_reduction': 0.10},
    'Prosperity-heavy':  {'honesty': 0.20, 'energy_clean': 0.15, 'abs_consumption': 0.10, 'prosperity': 0.40, 'co2_reduction': 0.15},
    'Reduction-heavy':   {'honesty': 0.20, 'energy_clean': 0.15, 'abs_consumption': 0.10, 'prosperity': 0.15, 'co2_reduction': 0.40},
}

scenario_ranks = {name: rank_with_weights(clean_scores_2000, scenario_weights)
                  for name, scenario_weights in scenarios.items()}
scenario_comparison = pd.DataFrame(scenario_ranks)
scenario_names = list(scenarios.keys())
scenario_comparison['best_rank']  = scenario_comparison[scenario_names].min(axis=1)
scenario_comparison['worst_rank'] = scenario_comparison[scenario_names].max(axis=1)
scenario_comparison['rank_range'] = (scenario_comparison['worst_rank']
                                     - scenario_comparison['best_rank'])
scenario_comparison = scenario_comparison.sort_values('Normative')

print("Rank under each named weighting scheme (2000 base, clean):")
print(scenario_comparison.head(12).to_string())

Rank under each named weighting scheme (2000 base, clean):
     Normative  Equal  Honesty-heavy  Destination-heavy  Prosperity-heavy  Reduction-heavy  best_rank  worst_rank  rank_range
PRT          1      4              1                  3                 6                4          1           6           5
SWE          2      1              3                  1                 2                1          1           3           2
FIN          3      2              2                  4                 3                2          2           4           2
NOR          4      3              4                  2                 1                7          1           7           6
GBR          5      5              5                  5                 5                3          3           5           2
EST          6      9              6                  9                11                6          6          11           5
DNK          7      6             11                  8    

In [14]:
# SENSITIVITY 2: Monte Carlo over the full weight space
# Draw random weightings (uniform over the simplex) and watch the ranking.
# N=2000 is statistically plenty: a proportion from N draws has std err ~ sqrt(p(1-p)/N).
weight_rng = np.random.default_rng(42)
N_WEIGHT_DRAWS = 2000
component_names = list(WEIGHTS.keys())
country_codes = clean_scores_2000['iso_code'].tolist()
rank_samples = {iso: [] for iso in country_codes}

for _ in range(N_WEIGHT_DRAWS):
    random_weights = dict(zip(component_names,
                              weight_rng.dirichlet(np.ones(len(component_names)))))
    for iso, rank in rank_with_weights(clean_scores_2000, random_weights).items():
        rank_samples[iso].append(rank)

mc_rows = []
for iso in country_codes:
    ranks = np.array(rank_samples[iso])
    mc_rows.append({
        'iso_code': iso,
        'median_rank': int(np.median(ranks)),
        'best': int(ranks.min()), 'worst': int(ranks.max()),
        'pct_top5':  round((ranks <= 5).mean() * 100, 1),
        'pct_top10': round((ranks <= 10).mean() * 100, 1),
    })
weight_mc = pd.DataFrame(mc_rows).sort_values('median_rank').reset_index(drop=True)

print(f"Monte Carlo sensitivity - {N_WEIGHT_DRAWS} random weightings (2000 base, clean):")
print(weight_mc.head(12).to_string(index=False))

# Rank-distribution box plot for the contenders (narrow box = robust regardless of weights)
contenders = weight_mc.head(12)['iso_code'].tolist()
rank_distribution = pd.DataFrame([{'iso_code': iso, 'rank': rank}
                                  for iso in contenders for rank in rank_samples[iso]])
fig = px.box(
    rank_distribution, x='iso_code', y='rank',
    category_orders={'iso_code': contenders},
    title=f'Rank stability across {N_WEIGHT_DRAWS} random weightings (2000 base, clean)<br>'
          '<sup>Narrow box = position holds no matter how you weight the components</sup>',
    labels={'rank': 'Rank', 'iso_code': ''}, height=520
)
fig.update_yaxes(autorange='reversed')  # rank 1 at top
fig.show()

# headline robustness statement
robust5  = weight_mc[weight_mc['pct_top5']  >= 80]['iso_code'].tolist()
robust10 = weight_mc[weight_mc['pct_top10'] >= 90]['iso_code'].tolist()
print(f"\nIn the top-5 under >=80% of ALL random weightings: {robust5}")
print(f"In the top-10 under >=90% of ALL random weightings: {robust10}")

Monte Carlo sensitivity - 2000 random weightings (2000 base, clean):
iso_code  median_rank  best  worst  pct_top5  pct_top10
     SWE            2     1      6      99.9      100.0
     FIN            3     1     21      87.4       98.5
     NOR            3     1     20      72.3       91.1
     PRT            4     1     15      76.8       94.1
     GBR            5     1     15      66.5       97.9
     DNK            6     1     23      40.8       89.2
     FRA            7     3     14      22.8       83.0
     ESP            8     3     13       9.8       89.2
     GRC           10     4     23       1.8       51.4
     EST           10     3     28       6.1       60.4
     DEU           11     4     27       2.0       41.1
     AUT           12     5     25       0.5       38.2



In the top-5 under >=80% of ALL random weightings: ['SWE', 'FIN']
In the top-10 under >=90% of ALL random weightings: ['SWE', 'FIN', 'NOR', 'PRT', 'GBR']


In [15]:
# FAIRNESS REFEREE 3: rank-correlation + leave-one-component-out
from scipy.stats import spearmanr

# (a) How similar are the rankings across the named scenarios? (1.0 = identical order)
scenario_rank_table = pd.DataFrame(scenario_ranks)
scenario_correlation = scenario_rank_table.corr(method='spearman')
pairwise_correlations = scenario_correlation.values[
    np.triu_indices_from(scenario_correlation.values, k=1)]
print("Spearman rank-correlation between weighting scenarios:")
print(scenario_correlation.round(2).to_string())
print(f"\nMean off-diagonal correlation: {pairwise_correlations.mean():.3f}  "
      f"(min {pairwise_correlations.min():.3f}) - close to 1.0 means the order barely depends on weights")

# (b) Leave-one-component-out: drop each component entirely, renormalize, compare to normative.
#     Low Spearman = that component is 'load-bearing' (the ranking leans on it).
normative_rank = rank_with_weights(clean_scores_2000, WEIGHTS)
clean_codes = clean_scores_2000['iso_code'].tolist()
normative_top5 = set(sorted(normative_rank, key=normative_rank.get)[:5])

print("\nLeave-one-component-out (drop a component, renormalize the other four):")
print(f"{'dropped':<16}{'spearman vs normative':>22}{'  top-5 change'}")
for dropped_component in WEIGHTS:
    loco_weights = {name: weight for name, weight in WEIGHTS.items()
                    if name != dropped_component}
    weight_total = sum(loco_weights.values())
    loco_weights = {name: weight / weight_total for name, weight in loco_weights.items()}
    loco_rank = rank_with_weights(clean_scores_2000, loco_weights)
    rho = spearmanr([normative_rank[iso] for iso in clean_codes],
                    [loco_rank[iso] for iso in clean_codes]).correlation
    loco_top5 = set(sorted(loco_rank, key=loco_rank.get)[:5])
    top5_changes = normative_top5.symmetric_difference(loco_top5)
    print(f"{dropped_component:<16}{rho:>22.3f}   "
          f"{sorted(top5_changes) if top5_changes else 'none'}")

Spearman rank-correlation between weighting scenarios:
                   Normative  Equal  Honesty-heavy  Destination-heavy  Prosperity-heavy  Reduction-heavy
Normative               1.00   0.94           0.99               0.97              0.92             0.91
Equal                   0.94   1.00           0.91               0.95              0.97             0.97
Honesty-heavy           0.99   0.91           1.00               0.95              0.88             0.88
Destination-heavy       0.97   0.95           0.95               1.00              0.90             0.89
Prosperity-heavy        0.92   0.97           0.88               0.90              1.00             0.96
Reduction-heavy         0.91   0.97           0.88               0.89              0.96             1.00

Mean off-diagonal correlation: 0.932  (min 0.879) - close to 1.0 means the order barely depends on weights

Leave-one-component-out (drop a component, renormalize the other four):
dropped          spearman vs 

In [16]:
# FAIRNESS REFEREE 4: threshold robustness
# Weights are not the only subjective choice - the SCORING anchors are too.
# Jitter every threshold by +-15% many times and check the ranking still holds.
import copy

jitter_rng = np.random.default_rng(7)
N_JITTER_RUNS = 200
JITTER_FRACTION = 0.15

def perturb_scoring(anchors, fraction, rng):
    jittered = copy.deepcopy(anchors)
    for metric, endpoints in jittered.items():
        for endpoint in ('zero', 'full'):
            jittered[metric][endpoint] = endpoints[endpoint] * (1 + rng.uniform(-fraction, fraction))
    return jittered

original_scoring = ACTIVE_SCORING   # build_scores re-reads ACTIVE_SCORING on every call
jitter_rank_samples = {iso: [] for iso in clean_scores_2000['iso_code']}

for _ in range(N_JITTER_RUNS):
    ACTIVE_SCORING = perturb_scoring(original_scoring, JITTER_FRACTION, jitter_rng)
    jittered_scores = clean_subset(build_scores(df, 2000)).reset_index(drop=True)
    for position, row in jittered_scores.iterrows():
        if row['iso_code'] in jitter_rank_samples:
            jitter_rank_samples[row['iso_code']].append(position + 1)
ACTIVE_SCORING = original_scoring

jitter_rows = []
for iso, ranks in jitter_rank_samples.items():
    ranks = np.array(ranks)
    jitter_rows.append({'iso_code': iso, 'median_rank': int(np.median(ranks)),
                        'best': int(ranks.min()), 'worst': int(ranks.max()),
                        'pct_top5': round((ranks <= 5).mean() * 100, 1)})
jitter_summary = pd.DataFrame(jitter_rows).sort_values('median_rank').reset_index(drop=True)

print(f"Threshold robustness - {N_JITTER_RUNS} runs, every anchor jittered +-{int(JITTER_FRACTION*100)}%:")
print(jitter_summary.head(12).to_string(index=False))
print("\n(If the top names here match the weight-based Monte Carlo, the ranking is robust")
print(" to BOTH subjective choices - weights AND thresholds.)")

Threshold robustness - 200 runs, every anchor jittered +-15%:
iso_code  median_rank  best  worst  pct_top5
     SWE            1     1      3     100.0
     PRT            2     1      4     100.0
     FIN            3     1      4     100.0
     NOR            4     2      4     100.0
     GBR            5     5      5     100.0
     EST            6     6      6       0.0
     DNK            7     7     10       0.0
     DEU            8     7      9       0.0
     ESP            9     7     10       0.0
     AUT           10     7     12       0.0
     GRC           11     8     12       0.0
     FRA           12    10     12       0.0

(If the top names here match the weight-based Monte Carlo, the ranking is robust
 to BOTH subjective choices - weights AND thresholds.)


In [17]:
# FAIRNESS REFEREE 5: component redundancy (construct validity)
# If two components correlate strongly, they double-count - one is partly redundant.
score_columns = ['s_honesty', 's_energy', 's_abs', 's_prosperity', 's_co2']
short_names = {'s_honesty': 'honesty', 's_energy': 'energy', 's_abs': 'abs_cons',
               's_prosperity': 'prosperity', 's_co2': 'co2_reduc'}

component_correlation = (scores_2000[score_columns].corr()
                         .rename(index=short_names, columns=short_names))
print("Component score correlation (|r|>0.6 = possible double-counting):")
print(component_correlation.round(2).to_string())

fig = px.imshow(component_correlation, text_auto='.2f',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Component correlation - are the 5 components measuring distinct things?',
                height=500)
fig.show()

correlation_pairs = [(short_names[col_a], short_names[col_b], component_correlation.iloc[i, j])
                     for i, col_a in enumerate(score_columns)
                     for j, col_b in enumerate(score_columns) if j > i]
redundant_pairs = [(name_a, name_b, round(corr_value, 2))
                   for name_a, name_b, corr_value in correlation_pairs if abs(corr_value) > 0.6]
print("\nStrongly-correlated pairs (|r|>0.6):",
      redundant_pairs if redundant_pairs else "none - components are distinct")

Component score correlation (|r|>0.6 = possible double-counting):
            honesty  energy  abs_cons  prosperity  co2_reduc
honesty        1.00    0.01      0.25        0.33       0.43
energy         0.01    1.00     -0.42        0.51       0.40
abs_cons       0.25   -0.42      1.00       -0.52      -0.44
prosperity     0.33    0.51     -0.52        1.00       0.56
co2_reduc      0.43    0.40     -0.44        0.56       1.00



Strongly-correlated pairs (|r|>0.6): none - components are distinct


In [ ]:
# FAIRNESS REFEREE 6: input-data uncertainty (error propagation)
# Consumption-based (MRIO) estimates carry roughly +-15-30% uncertainty in the literature
# (Owen 2017; Wiedmann & Lenzen 2018); territorial inventories are far tighter (~+-5%).
# Propagate that into the SCORE: jitter the input series N times, rebuild everything,
# and report each country's score as mean +- 95% CI and a rank INTERVAL, not a rank.
# Runtime: ~1-2 min (N_NOISE_RUNS full rebuilds, same engine as the threshold referee).

CBA_LEVEL_SD   = 0.12   # country-systematic consumption-accounting bias (the big one)
CBA_YEAR_SD    = 0.05   # year-to-year estimation noise on consumption
TERRITORIAL_SD = 0.03   # territorial inventory noise
N_NOISE_RUNS   = 300

noise_rng = np.random.default_rng(11)
score_samples      = {iso: [] for iso in scores_2000['iso_code']}
noisy_rank_samples = {iso: [] for iso in clean_scores_2000['iso_code']}

for _ in range(N_NOISE_RUNS):
    noisy_history = df.copy()
    for iso, row_index in noisy_history.groupby('iso_code').groups.items():
        country_bias = np.clip(noise_rng.normal(1, CBA_LEVEL_SD), 0.6, 1.4)
        noisy_history.loc[row_index, 'consumption_co2_per_capita'] *= country_bias * np.clip(
            noise_rng.normal(1, CBA_YEAR_SD, len(row_index)), 0.8, 1.2)
        noisy_history.loc[row_index, 'co2_per_capita'] *= np.clip(
            noise_rng.normal(1, TERRITORIAL_SD, len(row_index)), 0.9, 1.1)
    noisy_scores = build_scores(noisy_history, 2000)
    for _, row in noisy_scores.iterrows():
        if row['iso_code'] in score_samples:
            score_samples[row['iso_code']].append(row['final'])
    noisy_clean = clean_subset(noisy_scores).reset_index(drop=True)
    for position, row in noisy_clean.iterrows():
        if row['iso_code'] in noisy_rank_samples:
            noisy_rank_samples[row['iso_code']].append(position + 1)

uncertainty_rows = []
for iso in clean_scores_2000['iso_code']:
    score_arr = np.array(score_samples[iso])
    rank_arr  = np.array(noisy_rank_samples[iso])
    if len(score_arr) == 0:
        continue
    uncertainty_rows.append({
        'iso_code': iso,
        'score_mean': round(score_arr.mean(), 1),
        'score_sd':   round(score_arr.std(), 1),
        'ci_lo': round(np.percentile(score_arr, 2.5), 1),
        'ci_hi': round(np.percentile(score_arr, 97.5), 1),
        'rank_lo': int(np.percentile(rank_arr, 2.5)),
        'rank_hi': int(np.percentile(rank_arr, 97.5)),
    })
uncertainty = (pd.DataFrame(uncertainty_rows)
               .sort_values('score_mean', ascending=False).reset_index(drop=True))

print(f"Score uncertainty under input noise (CBA level +-{int(CBA_LEVEL_SD*100)}%, "
      f"year +-{int(CBA_YEAR_SD*100)}%, territorial +-{int(TERRITORIAL_SD*100)}%; {N_NOISE_RUNS} runs):")
print(uncertainty.head(12).to_string(index=False))

# who is statistically tied with the leader? (CI overlap)
leader = uncertainty.iloc[0]
tied_with_leader = uncertainty[(uncertainty['ci_hi'] >= leader['ci_lo']) &
                               (uncertainty['iso_code'] != leader['iso_code'])]
print(f"\nLeader: {leader['iso_code']} ({leader['score_mean']} "
      f"[{leader['ci_lo']}, {leader['ci_hi']}])")
print(f"Statistically tied with the leader (95% CI overlap): "
      f"{tied_with_leader['iso_code'].head(6).tolist()}")
print(" Near ties are TIES. The honest headline is a leading GROUP, not a single #1")

# Error-bar chart, top 12 by mean score
top12 = uncertainty.head(12)
fig = go.Figure(go.Scatter(
    x=top12['iso_code'], y=top12['score_mean'],
    error_y=dict(type='data', symmetric=False,
                 array=top12['ci_hi'] - top12['score_mean'],
                 arrayminus=top12['score_mean'] - top12['ci_lo']),
    mode='markers', marker=dict(size=10, color='#16a085')
))
fig.update_layout(
    title=f'TPI scores with 95% confidence bands - input uncertainty propagated ({N_NOISE_RUNS} runs)<br>'
          '<sup>Overlapping bands = statistically indistinguishable. MRIO data is soft; the index says so.</sup>',
    yaxis_title='TPI score (2000 base)', height=520
)
fig.show()

Score uncertainty under input noise (CBA level +-12%, year +-5%, territorial +-3%; 300 runs):
iso_code  score_mean  score_sd  ci_lo  ci_hi  rank_lo  rank_hi
     SWE        83.1       5.3   71.6   91.3        1        7
     PRT        82.5       1.9   76.8   84.9        1        5
     FIN        82.2       4.2   72.8   87.8        1        7
     NOR        80.7       3.4   73.5   86.3        1        6
     GBR        76.7       5.0   65.5   84.3        1       10
     EST        72.1       5.7   59.4   80.8        4       13
     DNK        68.2       6.4   55.8   78.9        4       14
     DEU        66.4       7.1   51.7   77.1        5       15
     ESP        66.2       7.8   49.8   80.0        4       15
     AUT        65.8       5.4   52.8   75.2        6       14
     GRC        65.3       6.4   52.1   76.8        5       15
     FRA        64.4       8.0   47.9   79.1        5       16

Leader: SWE (83.1 [71.6, 91.3])
Statistically tied with the leader (95% CI overlap): [

In [19]:
print("=" * 42)
print("NOTEBOOK 08 - TRANSITION PERFORMANCE INDEX")
print("KEY FINDINGS")
print("=" * 42)

top10 = clean_subset(scores_2000).head(10)
print("\nTop 10 (2000 base, normative weights, data-quality flags excluded):")
for position, (_, row) in enumerate(top10.iterrows(), 1):
    boundary_mark = ' *' if '*' in row['flag'] else ''
    print(f"  {position:>2}. {row['country']:<16}{boundary_mark:<3} {row['final']:6.1f}")

leader_row = clean_subset(scores_2000).iloc[0]
print(f"\nLeader: {leader_row['country']} ({leader_row['final']:.1f})")
print(f"   territorial -{leader_row['co2_red_pct']:.0f}%  |  consumption -{leader_row['cons_red_pct']:.0f}%  |  "
      f"GDP/cap +{leader_row['gdp_grw_pct']:.0f}% (${leader_row['gdp_latest']:,.0f})  |  "
      f"consumption {leader_row['cons_latest']:.1f} t/cap")

# face-validity: Switzerland should sink on honesty (can go negative)
for iso in ['CHE', 'POL', 'SWE', 'DEU', 'NOR']:
    if (scores_2000['iso_code'] == iso).any():
        row = scores_2000[scores_2000['iso_code'] == iso].iloc[0]
        rank = scores_2000.index[scores_2000['iso_code'] == iso][0] + 1
        print(f"   check {iso}: rank {rank:>2}{row['flag']:<2}, honesty {row['s_honesty']:6.1f}, "
              f"cons.reduction {row['cons_red_pct']:5.0f}%, divergence {row['divergence']:5.0f}")

# sufficiency headline (set by the overlay cell)
if 'sufficiency' in globals():
    on_track_count = sufficiency['verdict'].isin(['already at fair share', 'on track']).sum()
    print(f"\nSUFFICIENCY: {on_track_count}/{len(sufficiency)} countries are on a Paris-compatible path "
          f"({PARIS['target_t']}t/cap by {PARIS['target_year']}).")
    print("   High TPI rank = best in Europe, NOT fast enough in absolute terms.")

# uncertainty headline (set by Referee 6)
if 'uncertainty' in globals() and len(uncertainty):
    unc_leader = uncertainty.iloc[0]
    unc_tied = uncertainty[(uncertainty['ci_hi'] >= unc_leader['ci_lo']) &
                           (uncertainty['iso_code'] != unc_leader['iso_code'])]
    print(f"\nUNCERTAINTY: leader {unc_leader['iso_code']} = {unc_leader['score_mean']} "
          f"[{unc_leader['ci_lo']}, {unc_leader['ci_hi']}]; "
          f"statistically tied: {unc_tied['iso_code'].head(5).tolist()}")
    print("   The honest headline is a leading GROUP, not a single #1.")

print("""
Reading guide:
- '!' = data-quality flag (IRL/LUX/MLT) - excluded from the headline ranking.
- '*' = boundary flag (NOR petrostate) - kept, but its exported oil & gas emissions
        sit outside both territorial and consumption accounting.
- Robustness: 6 referees (scenarios, weight Monte Carlo, LOCO + rank correlation,
  threshold jitter, component redundancy, input-noise propagation).
- Sufficiency overlay and cumulative-responsibility context are NOT part of the score.
- Full methodology: 08_INDEX_METHODOLOGY.md
""")

NOTEBOOK 08 - TRANSITION PERFORMANCE INDEX
KEY FINDINGS

Top 10 (2000 base, normative weights, data-quality flags excluded):
   1. Sweden                83.6
   2. Portugal              83.2
   3. Finland               83.0
   4. Norway           *    81.0
   5. United Kingdom        77.7
   6. Estonia               73.6
   7. Denmark               69.3
   8. Germany               68.2
   9. Spain                 67.0
  10. Austria               66.6

Leader: Sweden (83.6)
   territorial -44%  |  consumption -38%  |  GDP/cap +38% ($47,125)  |  consumption 5.8 t/cap
   check CHE: rank 30  , honesty  -66.5, cons.reduction    -6%, divergence    46
   check POL: rank 20  , honesty   28.9, cons.reduction     8%, divergence     4
   check SWE: rank  1  , honesty   82.4, cons.reduction    38%, divergence     5
   check DEU: rank 10  , honesty   75.2, cons.reduction    32%, divergence     4
   check NOR: rank  4* , honesty   84.7, cons.reduction    34%, divergence    -7

SUFFICIENCY: 2/34 coun

In [20]:
# TRANSPARENCY: per-country scorecard
# Trace exactly how any country's score is built: raw input -> points -> weight -> contribution.

def explain(iso, scores=None, base_year=2000):
    scores = scores_2000 if scores is None else scores
    country_match = scores[scores['iso_code'] == iso]
    if country_match.empty:
        print(f"{iso} not in ranking"); return
    row = country_match.iloc[0]
    rank = scores.index[scores['iso_code'] == iso][0] + 1

    print(f"=== {row['country']} ({iso}) - rank {rank}, FINAL {row['final']:.1f}  (base {base_year}) ===")
    if '!' in row['flag']:
        print(f"  ! data-quality: {DATA_CAVEAT.get(iso, '')}")
    if '*' in row['flag']:
        print(f"  * boundary: {BOUNDARY_CAVEAT.get(iso, '')}")
    print(f"{'component':<16}{'raw input':>36}{'pts':>7}{'weight':>8}{'contrib':>8}")
    print('-' * 75)

    component_rows = [
        ('honesty',         f"cons -{row['cons_red_pct']:.0f}% | recent -{row['cons_red_recent_pct']:.0f}% | div {row['divergence']:+.0f}", row['s_honesty']),
        ('co2_reduction',   f"territorial -{row['co2_red_pct']:.0f}%",                  row['s_co2']),
        ('abs_consumption', f"{row['cons_latest']:.1f} t/cap now",                      row['s_abs']),
        ('energy_clean',    f"{row['en_latest']:.3f} kgCO2/kWh now",                    row['s_energy']),
        ('prosperity',      f"${row['gdp_latest']:,.0f}/cap | growth +{row['gdp_grw_pct']:.0f}%", row['s_prosperity']),
    ]
    available_weight = sum(WEIGHTS[name] for name, _, points in component_rows if pd.notna(points))
    for name, raw_input, points in component_rows:
        if pd.isna(points):
            print(f"{name:<16}{raw_input:>36}{'n/a':>7}{'-':>8}{'-':>8}")
            continue
        effective_weight = WEIGHTS[name] / available_weight  # renormalized if a component is n/a
        print(f"{name:<16}{raw_input:>36}{points:>7.1f}{effective_weight:>8.2f}{points*effective_weight:>8.1f}")
    print('-' * 75)
    print(f"{'composite':<16}{'(weighted sum above)':>36}{'':>7}{'':>8}{row['composite']:>8.1f}")
    print(f"{'momentum':<16}{'(consumption slope 2010-latest)':>36}{'':>7}{'':>8}{row['momentum']:>+8.1f}")
    print(f"{'FINAL':<16}{'':>36}{'':>7}{'':>8}{row['final']:>8.1f}")

# A star, a fake decoupler, the flagged petrostate, and the 2000-base underdog
for code in ['SWE', 'CHE', 'NOR', 'ROU']:
    explain(code)
    print()

=== Sweden (SWE) - rank 1, FINAL 83.6  (base 2000) ===
component                                  raw input    pts  weight contrib
---------------------------------------------------------------------------
honesty             cons -38% | recent -34% | div +5   82.4    0.40    33.0
co2_reduction                       territorial -44%   79.6    0.15    11.9
abs_consumption                        5.8 t/cap now   72.5    0.15    10.9
energy_clean                     0.061 kgCO2/kWh now   88.5    0.15    13.3
prosperity                 $47,125/cap | growth +38%   82.8    0.15    12.4
---------------------------------------------------------------------------
composite                       (weighted sum above)                   81.5
momentum             (consumption slope 2010-latest)                   +2.1
FINAL                                                                  83.6

=== Switzerland (CHE) - rank 30, FINAL 16.4  (base 2000) ===
component                                  raw 

## Export - `tpi_scores`

Writes the final TPI leaderboard (both base years) to Postgres so the dashboard Ch 4 reads it like any other query, and the future text-to-SQL agent can reach it. Re-run whenever the index is retuned.

In [ ]:
# Export the TPI leaderboard to Postgres (both base years, tagged).
# Dashboard Ch4 + the future agent read this; re-run after any retune.
tpi_2000 = scores_2000.copy(); tpi_2000['base_year'] = 2000
tpi_1990 = scores_1990.copy(); tpi_1990['base_year'] = 1990
tpi_scores = pd.concat([tpi_2000, tpi_1990], ignore_index=True)

tpi_scores.to_sql('tpi_scores', engine, if_exists='replace', index=False)
print(f"Exported {len(tpi_scores)} rows -> tpi_scores "
      f"({len(scores_2000)} @2000 + {len(scores_1990)} @1990)")